In [3]:
import numpy as np
import pandas as pd 

from scipy import stats

In [4]:
path="/home/dexter/Documents/GitHub/3D-textures/assets/experimental_corrected_results.csv"
df = pd.read_csv(path)
df.head()

,Printer,Texture,Test_Type,Comparison,Raw_Avg_Distance,Std_Deviation,Texture_Method_Error,Method_std_Error,Corrected_Distance
0,E,2,T,1 vs 2,0.000242,0.000202,0.000299,0.0,-5.783000e-05
1,E,2,T,1 vs 3,0.000267,0.000223,0.000299,0.0,-3.253000e-05
2,E,2,T,1 vs 4,0.000332,0.000337,0.000299,0.0,3.282000e-05
3,E,2,T,1 vs 5,0.000338,0.000265,0.000299,0.0,3.846000e-05
4,E,2,T,1 vs 6,0.000300,0.000236,0.000299,0.0,5.000000e-07


In [6]:
df.describe()

,Texture,Raw_Avg_Distance,Std_Deviation,Texture_Method_Error,Method_std_Error,Corrected_Distance
count,452.000000,452.000000,452.000000,452.000000,452.0,452.000000
mean,3.446903,0.000323,0.000290,0.000192,0.0,0.000131
std,1.718380,0.000136,0.000228,0.000060,0.0,0.000135
min,1.000000,0.000124,0.000073,0.000000,0.0,-0.000120
25%,2.000000,0.000235,0.000164,0.000168,0.0,0.000042
50%,3.000000,0.000281,0.000209,0.000191,0.0,0.000092
75%,5.000000,0.000386,0.000319,0.000208,0.0,0.000197
max,6.000000,0.001078,0.001847,0.000320,0.0,0.000848


In [ ]:

n_raw = 5
n_method = 5

se_raw = df["Std_Deviation"] / np.sqrt(n_raw)
se_method = df["Method_std_Error"] / np.sqrt(n_method)
df["SE_Corrected"] = np.sqrt(se_raw**2 + se_method**2)

# Welch's t-statistic, per row, from summary stats
df["t_Score"] = (df["Raw_Avg_Distance"] - df["Texture_Method_Error"]) / df["SE_Corrected"]

# Welch–Satterthwaite degrees of freedom, per row
df["df_welch"] = (se_raw**2 + se_method**2)**2 / (
    (se_raw**4) / (n_raw - 1) + (se_method**4) / (n_method - 1)
)

# One-tailed p-value: testing whether raw distance significantly exceeds method error
df["p_value"] = 1 - stats.t.cdf(df["t_Score"], df["df_welch"])

df["Corrected_Distance_um"] = df["Corrected_Distance"] * 1e6
df["SE_Corrected_um"] = df["SE_Corrected"] * 1e6

df["Formatted_Result"] = df.apply(
    lambda r: f"{r['Corrected_Distance_um']:.2f} ± {r['SE_Corrected_um']:.2f} μm",
    axis=1,
)

print(df[["Printer", "Comparison", "Formatted_Result", "t_Score", "p_value"]])


    Printer Comparison    Formatted_Result   t_Score   p_value
0         E     1 vs 2    45.74 ± 90.37 μm  0.506150  0.319687
1         E     1 vs 3   53.54 ± 102.30 μm  0.523339  0.314196
2         E     1 vs 4  154.40 ± 145.46 μm  1.061455  0.174160
3         E     1 vs 5  170.43 ± 128.04 μm  1.331050  0.126986
4         E     1 vs 6  177.83 ± 159.22 μm  1.116872  0.163303
..      ...        ...                 ...       ...       ...
447       B     6 vs 1  231.19 ± 146.90 μm  1.573784  0.095325
448       B     6 vs 2    97.28 ± 88.81 μm  1.095346  0.167446
449       B     6 vs 3    96.14 ± 96.16 μm  0.999793  0.186995
450       B     6 vs 4  202.22 ± 159.05 μm  1.271412  0.136241
451       B     6 vs 5    76.76 ± 91.71 μm  0.836944  0.224856

[452 rows x 5 columns]


In [10]:
alpha = 0.05
df["Significant"] = df["p_value"] < alpha
summary = (
    df.groupby(["Printer", "Texture"])
    .agg(
        N_Comparisons=("t_Score", "size"),
        Mean_Corrected_Distance_um=("Corrected_Distance_um", "mean"),
        Std_Corrected_Distance_um=("Corrected_Distance_um", "std"),
        Mean_t_Score=("t_Score", "mean"),
        Mean_p_value=("p_value", "mean"),
        N_Significant=("Significant", "sum"),
    )
    .reset_index()
)

summary["Success_Ratio_%"] = (
    100 * summary["N_Significant"] / summary["N_Comparisons"]
).round(1)

summary["Formatted_Result"] = summary.apply(
    lambda r: f"{r['Mean_Corrected_Distance_um']:.2f} ± {r['Std_Corrected_Distance_um']:.2f} μm",
    axis=1,
)

summary = summary.sort_values(["Printer", "Texture"]).reset_index(drop=True)

print(summary[
    ["Printer", "Texture", "N_Comparisons", "Formatted_Result",
     "Mean_t_Score", "Mean_p_value", "N_Significant", "Success_Ratio_%"]
])

# --- Optional: pivot to a Table-2-style layout (Printer rows, Texture columns) ---
pivot = summary.pivot(index="Printer", columns="Texture", values="Formatted_Result")
print("\nPivoted (paper-table style):")
print(pivot)

# --- Optional: same pivot but for success ratio, to see where noise-floor breakthroughs cluster ---
pivot_success = summary.pivot(index="Printer", columns="Texture", values="Success_Ratio_%")
print("\nSuccess ratio by texture:")
print(pivot_success)

   Printer  Texture  N_Comparisons    Formatted_Result  Mean_t_Score  \
0        B        1             30    95.33 ± 50.57 μm      0.927006   
1        B        2             30    68.58 ± 27.31 μm      0.762336   
2        B        3             30    77.58 ± 33.88 μm      0.871806   
3        B        4             30    72.78 ± 39.38 μm      0.911784   
4        B        5             30    74.78 ± 44.48 μm      0.836194   
5        B        6             30    30.51 ± 53.31 μm      0.351078   
6        E        1             30  229.24 ± 123.68 μm      1.077446   
7        E        2             30  177.74 ± 124.22 μm      1.144738   
8        E        3             20   330.37 ± 55.33 μm      2.544436   
9        E        4             30   221.65 ± 99.00 μm      1.266622   
10       E        5             30  259.27 ± 113.99 μm      1.343388   
11       E        6             20  417.08 ± 225.94 μm      1.165350   
12       R        1             20    37.26 ± 76.72 μm      0.22

In [ ]:
# 1. Calculate Mean Distance and Grand SE per printer
printer_summary = (
    df.groupby("Texture")
    .agg(
        Mean_Distance_um=("Corrected_Distance", lambda x: x.mean() * 1e6),
        Total_Tests=("p_value", "count"),
        Total_Tests=("p_value", "count"),
        # Count how many of the rows actually achieved statistical significance
        Significant_Tests=("p_value", lambda x: (x < 0.05).sum()),
    )
    .reset_index()
)

# 2. Compute the true Grand SE across the printer groups
sum_sq_se = (
    df.groupby("Printer")["SE_Corrected"]
    .apply(lambda x: np.sum(x**2))
    .reset_index()
)
printer_summary = printer_summary.merge(sum_sq_se, on="Texture")
printer_summary["Grand_SE_um"] = (
    np.sqrt(printer_summary["SE_Corrected"]) / printer_summary["Total_Tests"]
) * 1e6

# 3. Calculate the percentage of tests that broke through the noise floor
printer_summary["Significance_Rate"] = (
    printer_summary["Significant_Tests"] / printer_summary["Total_Tests"]
) * 100



# 4. Clean up formatting
printer_summary["Averaged_Geometry"] = printer_summary.apply(
    lambda r: f"{r['Mean_Distance_um']:.2f} ± {r['Grand_SE_um']:.2f} μm", axis=1
)
printer_summary["Success_Ratio"] = printer_summary.apply(
    lambda r: f"{int(r['Significant_Tests'])} / {int(r['Total_Tests'])} ({r['Significance_Rate']:.1f}%)",
    axis=1,
)

print(printer_summary[["Printer", "Total_Tests", "Averaged_Geometry", "Success_Ratio"]])

   Texture  Total_Tests  Averaged_Geometry    Success_Ratio
0        1           80  131.03 ± 17.44 μm    0 / 80 (0.0%)
1        2           80  143.79 ± 16.10 μm    7 / 80 (8.8%)
2        3           70  155.26 ± 14.32 μm  15 / 70 (21.4%)
3        4           72  131.14 ± 20.28 μm    0 / 72 (0.0%)
4        5           80  130.94 ± 15.59 μm    0 / 80 (0.0%)
5        6           70  171.80 ± 28.80 μm    1 / 70 (1.4%)
